# CS448 - Lab 4: 3D Audio


In this lab we will learn how to create 3D sounds for headphone playback. We will make use of simple filters and HRTFs to create static and moving sources. Use the three sounds ```fly.wav```, ```helicopter.wav```, and ```crumble.wav``` as sources for the 3D recording that you will create.

## Part 1: Static sources using ITD/ILD cues

Assume the sound source is two meter away from the center of your head (i.e., the middle point of the line that connects the two ears). The ears are 22.6 cm apart from each other. Directionality-wise, the source can be in one of the following source locations: 

- Straight ahead

- 45 degrees to the left

- 80 degrees to the right

- 135 degrees to the left

For each location find the source’s delay between the two ears, and design two filters that will simulate that ITD and ILD features (round the ITD delays to an integer sample size, i.e., your sample rate better be high enough). Assume that the direction of arrival angle on the left and right sides are the same, although they are slightly different. Below is an example figure of the first case. 

![alt text](45degrees.jpg)

When sounds come from the side of the head the attenuation at the contralateral ear is by a factor of 0.7. From sounds coming medial plane (between the ears) there will be no attenuation due to the head. For positions moving from the medial plane towards the sides you can interpolate between no attenuation and a factor of 0.7. Design and plot the filters that correspond to the locations shown above and use them to make 3D sounds with the three source signals.

Listen to the result through headphones and verify that they sound somewhat localized (it won’t sound perfect, but it should be believable). Do the 45 degrees left and 135 degrees left examples sound different?

In [18]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import scipy.signal as signal
# Make a sound player function that plays array "x" with a sample rate "rate", and labels it with "label"
def sound_player( x, rate=8000, label=''):
    from IPython.display import display, Audio, HTML
    display( HTML( 
    '<style> table, th, td {border: 0px; }</style> <table><tr><td>' + label + 
    '</td><td>' + Audio( x, rate=rate)._repr_html_()[3:] + '</td></tr></table>'
    ))

In [128]:
# YOUR CODE HERE
# raise NotImplementedError()

def interaural_filter(fs, azimuth, elevation=0):
    """
    Compute the interaural filter using the ITD and IID concept 
    for a given sampling frequency, azimuth and elevation.

    Parameters
    ----------
    fs : int
        Sampling frequency in Hz.
    azimuth : float
        Azimuth angle in degrees. 0 degrees is in front of the listener, 
        positive angles are to the left, and negative angles are to the right.
    elevation : float
        Elevation angle in degrees. default is 0. We don't use it for this problem.
    

    Returns
    -------
    h : numpy.ndarray
        Interaural filter.
    """

    # YOUR CODE HERE
    raise NotImplementedError()


# Load one of the signals.
# YOUR CODE HERE

# Get the filter with the desired azimuth.
# YOUR CODE HERE

# Check out the filters (left and right) by plotting them.
# YOUR CODE HERE

# Filter the signal with the left and right filters, and play the sound.
# YOUR CODE HERE

# Try other sources and azimuths, and listen to the results.


## Part 2. Static sources using HRTFs

In the ```hrtf``` directory you will find code for the function ```load_hrtf``` which returns the left and right HRTF filters given as input a source’s azimuth and elevation. These filters will be much better than the ITD/ILD filters for localizing sounds.

Apply the HRTFs on the given sources and create 3D sounds that correspond to the locations given above. For each source, you will beed to convolve it with the left and right HRTF of the desired position and generate two sounds, one for each channel. Verify that they sound correct using headphones; are they better than before? What differences do you observe? The left 45 degrees and left 135 degrees cases... Are they sounding different now? Try different elevation choices. Do you feel the difference?

Tip: When you use these make sure that the sample rates of the HRTFs and the sounds you convolve them with match.  The HRTFs are sampled at 44.1kHz.


In [129]:
import hrtf.load_hrtf as load_hrtf
import importlib
importlib.reload(load_hrtf)

# load one of the signals
# YOUR CODE HERE

# resample the signal to 44100 Hz if it's not already at that sample rate
# YOUR CODE HERE

L, R = load_hrtf.load_hrtf( ............... )

# Plot the filters
# YOUR CODE HERE

# Filter the signal with the left and right filters, and play the sound.
# YOUR CODE HERE



SyntaxError: invalid syntax. Perhaps you forgot a comma? (1279953203.py, line 11)

## Part 3. Dynamic Sources with HRTFs

In this part you will need to make a moving sound source. In order to do so we will make use of a fast convolution routine based on your STFT code from lab 1 (ha, you thought you were done with that!). In order to perform fast convolution we can perform an STFT of the sound to use, multiply each time frame of this transform with the DFT of the filter that we want  to impose and then use overlap add to transform back to the time domain.

Start by taking each sound from above, and apply your STFT on it. Make sure that the size of the transform is the same as the HRTF’s filter length. The hop size should be the same as the DFT size and you will need to zero pad each frame by as much as the DFT size in order to facilitate the tail of the convolution. Do not use an analysis/synthesis window.  Also zero pad the HRTF filters and take their DFT, so that you end up with an array which is the same size as each frame of the STFT.

Once you compute this STFT, go through its every time frame and element-wise multiply it with the desired HRTF filter to generate the STFT of the left and right sounds. Figure out which HRTF angle to multiply each frame with so that by the end of the sound you will have made it go around your head.

Once you perform these operations you will have generated two STFT matrices, one for the left channel and one for the right. Use your inverse STFT routine and play the stereo sound through your headphones. You should hear a convincing rendering of the original sounds circling around your head.

In [131]:
# Define the stft function. Note that you want use a frame size smaller than the dft size, 
# and pad the frame with zeros to get the dft size. This way, you can get a smoother spectrogram.
# You might not have implemented the zero padding part before. 
# Do not apply windowing.

def stft( input_sound, frame_size, dft_size, hop_size, window=None):
    """Compute the short-time Fourier transform of a sound.
    Args:
        input_sound: 1D numpy array containing the sound samples
        frame_size: Number of effective samples in each frame (before padding)
        dft_size: Size of the DFT to compute (number of frequency bins; must be >= frame_size)
        hop_size: Number of samples to advance between successive frames 
        window: 1D numpy array containing the analysis window (length = dft_size; default=None, which means no windowing)
    Returns:
        f: 2D numpy array (frequencies x time) containing the complex-valued STFT
    """
    # # YOUR CODE HERE
    # raise NotImplementedError()


# Define the istft function. This should be the inverse of the stft function you defined above.
# Do not apply windowing.

def istft( stft_output, frame_size, dft_size, hop_size, window=None):
    """Compute the inverse short-time Fourier transform of a spectrogram.
    Args:
        stft_output: 2D numpy array (frequencies x time) containing the complex-valued STFT
        frame_size: Number of effective samples in each frame (before padding)
        dft_size: Size of the DFT that was computed (number of frequency bins)
        hop_size: Number of samples advanced between successive frames
        window: 1D numpy array containing the synthesis window (length = dft_size)
    Returns:
        x: 1D numpy array containing the reconstructed sound samples
    """
    # YOUR CODE HERE

# Come up with the right frame, dft, and hop sizes. 
# YOUR CODE HERE

# Calculate spectrogram from the source
# YOUR CODE HERE


# Sample from linearly moving (azimuth, elevation) and call the load_hrtf.load_hrtf function 
# to get the corresponding filters, and apply the filters to the columns of spectrogram, respectively.
# YOUR CODE HERE

# Reconstruct the sound using the istft function, and play the sound.
# YOUR CODE HERE